In [ ]:
from services.chatbot import ChatModel
from huggingface_hub import InferenceClient, login
from services.chatbot import ChatChain
from transformers import AutoTokenizer
from services.extract_data import get_data
from clients.prediction_model import getPrediction
from prompts import system_prompt
# from clients.alpha_vantage import fetch_monthly_minute_data
import pandas as pd
from services.chatbot import mintly
import json 
import numpy as np

c:\Users\mishr\OneDrive\Desktop\DOT-SLASH-\mintzy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
system_prompt = """I have time-series data for a stock listing. An example stock listing for a certain label looks like this:

{
  "ticker": "INFY",
  "data": {
    "uid": "1",
    "datetime": "2025-01-02 04:00:00",
    "open": 22.71,
    "high": 22.9,
    "low": 22.71,
    "close": 22.9,
    "volume": 944.0,
    "EMA": 22.9,
    "Volume_Oscillator": 0.0,
    "RSI": 50.21927551695059,
    "%K": 100.0,
    "%D": 100.0,
    "+DI": 22.653346947797505,
    "-DI": 24.24242424242359,
    "ADX": 30.58052962628356,
    "PVT": -15111.818938244143,
    "%Change": -3.4346103038309166,
    "RRR": 0.2565445026178002
  }
}

There are some questions in which you have to predict(Using a model) some data on the basis of the labels and time frame, ALWAYS assume it to be NULL.
To do this, you should provide the following action in the format:

## Action
predict[{
    "given": {
        "ticker": [<ticker1>, <ticker2>, ...], 
        "time_frame": {NULL}
    },
    required: ["prediction"]
}]

If you find all the information you needed from the observation I provided, then you give the action:
        
    finish[<whatever-answer-you-concluded-from-the-observation>]

 VERY IMPORTANT: YOU SHOULD ALWAYS STOP GENERATING TEXT AFTER GIVING ACTION FOR THE OBSERVATION THAT THE USER WILL PROVIDE. Also, you should expect observation only when previous action is the search action.

Now, I want the whole process in the following format:
## Question: <whatever-my-question-is>
## Thought: <your-thoughts-and-action-plan-based-on-my-question-and-observation-i-provided>
## Action: <either-search-action-or-finish-action-in-the-format-i-provided>
## Observation: <whatever-observation-I-WILL-PROVIDE>
## Thought: <your-thoughts-and-action-plan-based-on-observation-i-provided>
## Action: <either-search-action-or-finish-action-in-the-format-i-provided>
... until the action is finish.

Remark: In the process format, I WILL provide the question as ## Question. You are NOT SUPPOSED to again give the ## Question in the tests you will generate.

Following is an example run that show the process:

##Question: Can you give me analysis for Apple in real time?
##Thought: The user is asking for a real-time analysis of Apple (AAPL). Since this requires the latest data, I will perform a search to retrieve up-to-date time-series stock information for AAPL.
##Action:

predict[{
"given": {
"ticker": ["AAPL"],
"time_frame": {NULL}
},
"required": ["prediction"]
}]

##Observation: "The percentage change in the stock price is over the duration is 5%."
##Thought: The observation indicates that Apple's stock price has changed by 5% over the given duration. Based on this, I can conclude that the stock has experienced significant movement, which may indicate strong momentum or volatility.
Action:

finish["Apple's stock has experienced a 5% change over the observed duration, suggesting notable market movement. Further analysis may be required to determine trends, resistance levels, or future projections."]
"""

In [8]:
system_msg = {"role" : "system", "content" : system_prompt}
chatChain = ChatChain(chain=[system_msg])
mintly = mintly(chatChain=chatChain)

output = mintly.chat("Can you give me analysis for Infosys in real time?")

68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [9]:
output

'"Infosys\' stock price has experienced a 3.59% change over the observed duration, suggesting a moderate level of market activity. Further analysis would be beneficial to determine the specific drivers behind this movement and potential future trends."'

In [14]:
output = mintly.chat("Can you give me analysis for Reliance in real time?")
output

Error: Label 'RELIANCE' not found in model dictionary.


'"Reliance\'s stock price shows no change (0%) over the observed duration, indicating a period of stability or low market activity. Further investigation into the trading volume and broader market context might provide additional insights."'

In [15]:
for msg in chatChain.chain :  print(msg, "\n")

{'role': 'system', 'content': 'I have time-series data for a stock listing. An example stock listing for a certain label looks like this:\n\n{\n  "ticker": "INFY",\n  "data": {\n    "uid": "1",\n    "datetime": "2025-01-02 04:00:00",\n    "open": 22.71,\n    "high": 22.9,\n    "low": 22.71,\n    "close": 22.9,\n    "volume": 944.0,\n    "EMA": 22.9,\n    "Volume_Oscillator": 0.0,\n    "RSI": 50.21927551695059,\n    "%K": 100.0,\n    "%D": 100.0,\n    "+DI": 22.653346947797505,\n    "-DI": 24.24242424242359,\n    "ADX": 30.58052962628356,\n    "PVT": -15111.818938244143,\n    "%Change": -3.4346103038309166,\n    "RRR": 0.2565445026178002\n  }\n}\n\nThere are some questions in which you have to predict(Using a model) some data on the basis of the labels and time frame, ALWAYS assume it to be NULL.\nTo do this, you should provide the following action in the format:\n\n## Action\npredict[{\n    "given": {\n        "ticker": [<ticker1>, <ticker2>, ...], \n        "time_frame": {NULL}\n    }

In [11]:
# message = "[The percentage change in the stock price is over the duration is 2 %.]"

In [12]:
# def identify_msg_type(input_string: str) -> str:

#         trimmed_string = input_string.strip()
#         if trimmed_string.startswith('[') and trimmed_string.endswith(']'):
#             return "Observation"

#         return "Question"

In [13]:
# response = identify_msg_type(message)
# response